# 01 Data Preprocessing Riddhiman R

Data preprocessing notebook (R)

Purpose: early cohort assembly and preprocessing in the Truveta environment.

Notes: this notebook was copied from the original project files, outputs were cleared for repository use, and obvious hard-coded study IDs were replaced with placeholders where possible.

# Initialize Truveta SDK

In [ ]:
library(readr, warn.conflicts = FALSE)
library(arrow, warn.conflicts = FALSE)
library(magrittr, warn.conflicts = FALSE)
library(stringr, warn.conflicts = FALSE)
library(dplyr, warn.conflicts = FALSE)
library(rlang, warn.conflicts = FALSE)
library(data.table, warn.conflicts = FALSE)
library(lubridate, warn.conflicts = FALSE)
library(tidyr, warn.conflicts = FALSE)
library(truveta.notebook.study)

In [ ]:
# Use only one statement below and comment out whichever you are not using.
# con <- create_connection(output_mode = "sparklyr")
con <- create_connection(output_mode = "sparkr")

In [ ]:
study <- get_study(con)
# Use only one statement below and comment out whichever you are not using.
# population <- get_population(con, study, title='Test_Luke')
population <- get_population(con, id='<TRUVETA_POPULATION_ID>')

population

In [ ]:
# Get latest completed active snapshot.
snapshot = get_latest_snapshot(con, population)
snapshot

In [ ]:
# Show tables in the snapshot.
get_tables(con, snapshot)

In [ ]:
sdoh_data<- load_artifacts_data(con, study, "/data/SDOH_DATA_ALL_10_18_2024.parquet")
display_df(sdoh_data, 5)

cohort_data<- load_artifacts_data(con, study, "/data/PREVENT_COHORT_10_16_2024.parquet")
display_df(cohort_data, 5)

In [ ]:
library(dplyr)
cohort_data_1<-cohort_data %>% collect()
sdoh_data_1<-sdoh_data %>% collect()

In [ ]:
print(nrow(cohort_data_1))

print(nrow(sdoh_data_1))

In [ ]:
final_df_raw <- cohort_data %>%
  inner_join(sdoh_data, by = "PersonId")

display_df(final_df_raw)

In [ ]:
# Convert index_dtt to year

final_data<-  final_df_raw %>%
  mutate(index_dtt = to_timestamp(index_dtt),
         year = year(index_dtt),
         age=as.numeric(age),
         scre=as.numeric(scre))


In [ ]:
print(names(final_data))

In [ ]:
final_data=final_data %>% collect()
final_data <- final_data[!is.na(final_data$female), ]
scre = final_data$scre
age = final_data$age
female = final_data$female


egfr_female = 141.57 * pmin(scre / 0.7, 1) ^ -0.241 * pmax(scre / 0.7, 1) ^ -1.2 * 0.993839 ^ age
egfr_male = 141.57 * pmin(scre / 0.9, 1) ^-0.302 * pmax(scre / 0.9, 1) ^ -1.2 * 0.993839 ^ age
final_data$egfr=ifelse(female == 0, egfr_male, egfr_female)

In [ ]:
final_data$fu_ascvd =pmin(final_data$fu_mi, final_data$fu_str)
final_data$fu_cvd = pmin(final_data$fu_mi, final_data$fu_str, final_data$fu_hf)

In [ ]:
final_data$age = (final_data$age - 55) / 10
final_data$nonhdlc = (final_data$chol - final_data$hdlc) * 0.02586 - 3.5
final_data$hdlc = (final_data$hdlc * 0.02586 - 1.3) / 0.3
final_data$sbp1 = (pmin(final_data$sbp, 110) - 110) / 20
final_data$sbp2 = (pmax(final_data$sbp, 110) - 130) / 20
final_data$bmi1 = (pmin(final_data$bmi, 30) - 25) / 5
final_data$bmi2 = (pmax(final_data$bmi, 30) - 30) / 5
final_data$egfr1 = (pmin(final_data$egfr, 60) - 60) / -15
final_data$egfr2 = (pmax(final_data$egfr, 60) - 90) / -15

In [ ]:
hist(final_data$age)

In [ ]:
final_data$sbp2treat= final_data$sbp2 * final_data$antihtn
final_data$nonhdlctreat = final_data$nonhdlc * final_data$statin
final_data$agenonhdlc = final_data$age * final_data$nonhdlc
final_data$agehdlc = final_data$age * final_data$hdlc
final_data$agesbp2 = final_data$age * final_data$sbp2
final_data$agediabetes = final_data$age * final_data$diabetes
final_data$agecursmk = final_data$age * final_data$cursmk
final_data$agebmi2 = final_data$age * final_data$bmi2
final_data$ageegfr1 = final_data$age * final_data$egfr1
final_data$cons = 1

In [ ]:
install.packages("mice")

In [ ]:
#,"Attribute_Value_Numeric_ProspectEstimatedIncomeRange"
#,"Attribute_Value_Numeric_AddressChangeCountLast60Months"
#"Attribute_Value_Categorical_HouseholdAnnualIncomeRange",
#"Attribute_Value_Numeric_FirstDegreeRelativesCount",
#"Attribute_Value_Categorical_HighestEduIndividual",
#"Attribute_Value_Categorical_DistanceToCloseTies",
#,"Attribute_Value_Numeric_CurrAddrLenOfRes"
local_final_data=collect(final_data)
impute_columns=c("Attribute_Value_Categorical_CurrAddrDwellType","Attribute_Value_Numeric_CurrAddrMedianIncome","Attribute_Value_Numeric_RaAMmbrCnt")
data_to_impute=local_final_data[,impute_columns]
#print(data_to_impute)



In [ ]:
library(mice)
imputed_data=mice(data_to_impute,m=1,method="pmm",seed=321)

In [ ]:
completed_impute<-complete(imputed_data)
display_df(completed_impute)

In [ ]:
#print(sum(is.na(completed_impute$Attribute_Value_Numeric_ProspectEstimatedIncomeRange)))
#Attribute_Value_Categorical_HouseholdAnnualIncomeRange
#print(sum(is.na(completed_impute$Attribute_Value_Categorical_HouseholdAnnualIncomeRange)))
#Attribute_Value_Numeric_AddressChangeCountLast60Months
#print(sum(is.na(completed_impute$Attribute_Value_Numeric_AddressChangeCountLast60Months)))
#Attribute_Value_Categorical_CurrAddrDwellType
print(sum(is.na(completed_impute$Attribute_Value_Categorical_CurrAddrDwellType)))

#"Attribute_Value_Numeric_CurrAddrMedianIncome"
#"Attribute_Value_Numeric_RaAMmbrCnt"

print(sum(is.na(completed_impute$Attribute_Value_Numeric_CurrAddrMedianIncome)))


In [ ]:
imputed_data$loggedEvents
print(length(completed_impute$Attribute_Value_Numeric_CurrAddrMedianIncome))
print(nrow(final_data))

temp_data=cbind(final_data$Person_Id,completed_impute$Attribute_Value_Numeric_CurrAddrMedianIncome)

final_data$Attribute_Value_Numeric_CurrAddrMedianIncome=completed_impute$Attribute_Value_Numeric_CurrAddrMedianIncome


print(sum(is.na(final_data$cvd)))
print(sum(is.na(final_data$age)))
print(sum(is.na(final_data$nonhdlc)))
print(sum(is.na(final_data$sbp1)))
print(sum(is.na(final_data$hdlc)))
print(sum(is.na(final_data$sbp2)))
print(sum(is.na(final_data$diabetes)))
print(sum(is.na(final_data$egfr1)))
print(sum(is.na(final_data$egfr2)))
print(sum(is.na(egfr_male)))




In [ ]:
#hist(final_data$sbp1)
print(table(final_data$sbp1==0))

In [ ]:
library(survival)

In [ ]:
final_data_1=final_data %>% collect()

#print(names(final_data_1[,24:28]))

In [ ]:
#sdoh variables
sdoh_variables=c("Attribute_Value_Numeric_FirstDegreeRelativesCount","Attribute_Value_Numeric_AddressChangeCountLast60Months", "Attribute_Value_Categorical_HighestEduIndividual","Attribute_Value_Categorical_MedicationNonAdherenceRiskCategory","Attribute_Value_Categorical_CurrentAddressOwnOrRent","Attribute_Value_Categorical_CollegePastPresent","Attribute_Value_Categorical_CurrAddrDwellType","Attribute_Value_Categorical_HealthManagementNonMotivationRiskCategory","Attribute_Value_Categorical_HighestEduHousehold","Attribute_Value_Numeric_CurrAddrMedianIncome","Attribute_Value_Numeric_MedicationAdherenceRate","Attribute_Value_Numeric_HealthManagementMotivationLevel","Attribute_Value_Categorical_HealthcareCostRiskCategory")
length(sdoh_variables)

In [ ]:
final_data_missing <- data.frame(matrix(0, nrow(final_data_1), 13))

names(final_data_missing) <- sdoh_variables

print(names(final_data_missing))

In [ ]:
for(i in 1:nrow(final_data_1)){
   for(j in 1:length(sdoh_variables)) {
    final_data_missing[i, sdoh_variables[j]] <- 1*(is.na(final_data_1[i, sdoh_variables[j]])||is.null(final_data_1[i, sdoh_variables[j]]))
}
}


print(sum(is.na(final_data_1$Attribute_Value_Numeric_FirstDegreeRelativesCount)))
print(nrow(final_data_1))

In [ ]:
display_df(final_data_missing)

##sdoh columns
#sdoh_variables=c("Attribute_Value_Numeric_FirstDegreeRelativesCount","Attribute_Value_Numeric_AddressChangeCountLast60Months", "Attribute_Value_Categorical_HighestEduIndividual","Attribute_Value_Categorical_MedicationNonAdherenceRiskCategory","Attribute_Value_Categorical_CurrentAddressOwnOrRent","Attribute_Value_Categorical_CollegePastPresent","Attribute_Value_Categorical_CurrAddrDwellType","Attribute_Value_Categorical_HealthManagementNonMotivationRiskCategory","Attribute_Value_Categorical_HighestEduHousehold","Attribute_Value_Numeric_CurrAddrMedianIncome","Attribute_Value_Numeric_MedicationAdherenceRate","Attribute_Value_Numeric_HealthManagementMotivationLevel","Attribute_Value_Categorical_HealthcareCostRiskCategory")

##impute Values
#continuous
impute_value_FirstDegreeRelativesCount=mean(final_data_1$Attribute_Value_Numeric_FirstDegreeRelativesCount[final_data_missing$Attribute_Value_Numeric_FirstDegreeRelativesCount==0])
impute_value_AddressChangeCountLast60Months=mean(final_data_1$Attribute_Value_Numeric_AddressChangeCountLast60Months[final_data_missing$Attribute_Value_Numeric_AddressChangeCountLast60Months==0])
impute_value_CurrAddrMedianIncome=mean(final_data_1$Attribute_Value_Numeric_CurrAddrMedianIncome[final_data_missing$Attribute_Value_Numeric_CurrAddrMedianIncome==0])
impute_value_MedicationAdherenceRate=mean(final_data_1$Attribute_Value_Numeric_MedicationAdherenceRate[final_data_missing$Attribute_Value_Numeric_MedicationAdherenceRate==0])
impute_value_HealthManagementMotivationLevel=mean(final_data_1$Attribute_Value_Numeric_HealthManagementMotivationLevel[final_data_missing$Attribute_Value_Numeric_HealthManagementMotivationLevel==0])





In [ ]:
# Categorical
impute_value_Attribute_Value_Categorical_HighestEduIndividual <- 
    names(which.max(table(final_data_1$Attribute_Value_Categorical_HighestEduIndividual[
        final_data_missing$Attribute_Value_Categorical_HighestEduIndividual == 0])))

impute_value_Attribute_Value_Categorical_MedicationNonAdherenceRiskCategory <- 
    names(which.max(table(final_data_1$Attribute_Value_Categorical_MedicationNonAdherenceRiskCategory[
        final_data_missing$Attribute_Value_Categorical_MedicationNonAdherenceRiskCategory == 0])))

impute_value_Attribute_Value_Categorical_CurrentAddressOwnOrRent <- 
    names(which.max(table(final_data_1$Attribute_Value_Categorical_CurrentAddressOwnOrRent[
        final_data_missing$Attribute_Value_Categorical_CurrentAddressOwnOrRent == 0])))

impute_value_Attribute_Value_Categorical_CollegePastPresent <- 
    names(which.max(table(final_data_1$Attribute_Value_Categorical_CollegePastPresent[
        final_data_missing$Attribute_Value_Categorical_CollegePastPresent == 0])))

impute_value_Attribute_Value_Categorical_CurrAddrDwellType <- 
    names(which.max(table(final_data_1$Attribute_Value_Categorical_CurrAddrDwellType[
        final_data_missing$Attribute_Value_Categorical_CurrAddrDwellType == 0])))

impute_value_Attribute_Value_Categorical_HealthManagementNonMotivationRiskCategory <- 
    names(which.max(table(final_data_1$Attribute_Value_Categorical_HealthManagementNonMotivationRiskCategory[
        final_data_missing$Attribute_Value_Categorical_HealthManagementNonMotivationRiskCategory == 0])))

impute_value_Attribute_Value_Categorical_HighestEduHousehold <- 
    names(which.max(table(final_data_1$Attribute_Value_Categorical_HighestEduHousehold[
        final_data_missing$Attribute_Value_Categorical_HighestEduHousehold == 0])))

impute_value_Attribute_Value_Categorical_HealthcareCostRiskCategory <- 
    names(which.max(table(final_data_1$Attribute_Value_Categorical_HealthcareCostRiskCategory[
        final_data_missing$Attribute_Value_Categorical_HealthcareCostRiskCategory == 0])))

In [ ]:
final_data_1$Attribute_Value_Numeric_FirstDegreeRelativesCount[final_data_missing$Attribute_Value_Numeric_FirstDegreeRelativesCount==1]=impute_value_FirstDegreeRelativesCount
final_data_1$Attribute_Value_Numeric_AddressChangeCountLast60Months[final_data_missing$Attribute_Value_Numeric_AddressChangeCountLast60Months==1]=impute_value_AddressChangeCountLast60Months
final_data_1$Attribute_Value_Numeric_CurrAddrMedianIncome[final_data_missing$Attribute_Value_Numeric_CurrAddrMedianIncome==1]=impute_value_CurrAddrMedianIncome
final_data_1$Attribute_Value_Numeric_MedicationAdherenceRate[final_data_missing$Attribute_Value_Numeric_MedicationAdherenceRate==1]=impute_value_MedicationAdherenceRate
final_data_1$Attribute_Value_Numeric_HealthManagementMotivationLevel[final_data_missing$Attribute_Value_Numeric_HealthManagementMotivationLevel==1]=impute_value_HealthManagementMotivationLevel
final_data_1$Attribute_Value_Categorical_HighestEduIndividual[final_data_missing$Attribute_Value_Categorical_HighestEduIndividual==1]=impute_value_Attribute_Value_Categorical_HighestEduIndividual
final_data_1$Attribute_Value_Categorical_MedicationNonAdherenceRiskCategory[final_data_missing$Attribute_Value_Categorical_MedicationNonAdherenceRiskCategory==1]=impute_value_Attribute_Value_Categorical_MedicationNonAdherenceRiskCategory
final_data_1$Attribute_Value_Categorical_CurrentAddressOwnOrRent[final_data_missing$Attribute_Value_Categorical_CurrentAddressOwnOrRent==1]=impute_value_Attribute_Value_Categorical_CurrentAddressOwnOrRent
final_data_1$Attribute_Value_Categorical_CollegePastPresent[final_data_missing$Attribute_Value_Categorical_CollegePastPresent==1]=impute_value_Attribute_Value_Categorical_CollegePastPresent
final_data_1$Attribute_Value_Categorical_CurrAddrDwellType[final_data_missing$Attribute_Value_Categorical_CurrAddrDwellType==1]=impute_value_Attribute_Value_Categorical_CurrAddrDwellType
final_data_1$Attribute_Value_Categorical_HealthManagementNonMotivationRiskCategory[final_data_missing$Attribute_Value_Categorical_HealthManagementNonMotivationRiskCategory==1]=impute_value_Attribute_Value_Categorical_HealthManagementNonMotivationRiskCategory
final_data_1$Attribute_Value_Categorical_HighestEduHousehold[final_data_missing$Attribute_Value_Categorical_HighestEduHousehold==1]=impute_value_Attribute_Value_Categorical_HighestEduHousehold
final_data_1$Attribute_Value_Categorical_HealthcareCostRiskCategory[final_data_missing$Attribute_Value_Categorical_HealthcareCostRiskCategory==1]=impute_value_Attribute_Value_Categorical_HealthcareCostRiskCategory



In [ ]:
print(sum(is.na(final_data_1$Attribute_Value_Numeric_FirstDegreeRelativesCount)))
print(nrow(final_data_1))